# **Задача Авито**

**Идея решения:** сначала мы собираем подходящие объявления с помощью `E5`, `BM25` и символьного `TF-IDF`. Затем `LightGBM` сортирует кандидатов по 54 признакам, и мы оставляем лучшие 50.

**Что делает ноутбук:**

- загружает историю взаимодействий, каталог объявлений и поисковые запросы;
- находит кандидатов по смыслу текста и совпадениям слов;
- учитывает географию, фильтры и историю похожих запросов;
- обучает классификатор услуг и ранжировщик `LightGBM`;
- проверяет локальное качество и сохраняет `answer.csv`.

**P.S.** Ноутбук рассчитан на платформу `Kaggle`: выбираем **GPU T4 ×2** и подключаем датасет **AvitoDataset**.


### 1. Импорты и настройки

Тут мы подключаем библиотеки, задаём пути к данным и параметры расчёта. В `DATA` должны лежать три файла:

- `train.parquet` - история положительных взаимодействий пользователей с объявлениями;
- `benchmark_items.parquet` - каталог объявлений для итогового ответа;
- `benchmark_queries.parquet` - запросы, для которых нужно найти объявления.

Если путь к датасету в `Kaggle` отличается, меняем `DATA` в ячейке ниже.

| Параметр | Значение | Смысл |
|---|---|---|
| `MODEL` | `intfloat/multilingual-e5-base` | модель для векторного представления текстов |
| `EMB_BATCH` | 256 | сколько текстов кодируем за один шаг |
| `QUERY_BATCH` | 256 | сколько запросов обрабатываем при поиске кандидатов |
| `OUT` | `/kaggle/working/artifacts` | папка для моделей, векторов и метрик |


In [1]:
from functools import lru_cache
import scipy.sparse as sp
from nltk.stem.snowball import RussianStemmer
import json
import re
import time
import unicodedata
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download

SEED = 42
SPLIT_SEED = 43
EMB_BATCH = 256
QUERY_BATCH = 256

MODEL = 'intfloat/multilingual-e5-base'

DATA = Path('/kaggle/input/datasets/btwdanik/avitodataset')
MODEL_PATH = Path('/kaggle/working/models/multilingual-e5-base')
OUT = Path('/kaggle/working/artifacts')

OUT.mkdir(parents=True, exist_ok=True)

SEARCH = [
    'search_query',
    'search_location_id',
    'search_is_delivery_search',
    'search_infm_params_text',
    'search_category'
]

N_GPU = torch.cuda.device_count()
DEVICE = torch.device("cuda:0" if N_GPU else "cpu")
GPU_IDS = list(range(min(N_GPU, 2)))
SECOND_DEVICE = "cuda:1" if N_GPU > 1 else DEVICE

for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i))

torch.set_num_threads(6)

print("=" * 50)
print("CONFIG")
print("=" * 50)

print(f"Device      : {DEVICE}")
print(f"GPUs        : {N_GPU}")

for i in range(N_GPU):
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}    : {torch.cuda.get_device_name(i)} ({vram:.1f} GB)")

print(f"EMB_BATCH   : {EMB_BATCH}")
print(f"QUERY_BATCH : {QUERY_BATCH}")
print(f"Model       : {MODEL}")
print(f"Data        : {DATA}")
print(f"Output      : {OUT}")

print("=" * 50)

FEATURES = [
    'bm25_title', 'bm25_body', 'char_title', 'bm25_filter', 'e5_body', 'e5_title', 'lexical',
    'hybrid', 'history', 'history_normalized', 'semantic_category', 'exact_location',
    'log_distance_km', 'location_lift', 'location_probability', 'geo_bias', 'popularity', 'rating',
    'log_reviews', 'log_price', 'phone_hidden', 'message_forbidden', 'category_matches',
    'microcategory', 'title_length', 'description_length', 'title_coverage', 'title_precision',
    'query_in_title', 'bm25_title_normalized', 'bm25_body_normalized', 'bm25_filter_normalized',
    'char_title_normalized', 'nearest_query_similarity', 'query_locality', 'geo_known',
    'query_tokens', 'query_length', 'has_filter', 'query_seen', 'query_category',
    'body_term_coverage', 'body_idf_coverage', 'body_all_terms', 'query_in_body',
    'service_probability', 'service_probability_relative', 'service_type_match',
    'service_subtype_match', 'rating_filter_pass', 'numbers_in_title', 'semantic_gap_body',
    'semantic_gap_title', 'service_entropy',
]


0 Tesla T4
1 Tesla T4
CONFIG
Device      : cuda:0
GPUs        : 2
  GPU 0    : Tesla T4 (14.6 GB)
  GPU 1    : Tesla T4 (14.6 GB)
EMB_BATCH   : 256
QUERY_BATCH : 256
Model       : intfloat/multilingual-e5-base
Data        : /kaggle/input/datasets/btwdanik/avitodataset
Output      : /kaggle/working/artifacts


### 2. Загрузка модели E5

Загружаем `multilingual-e5-base` и токенизатор из `Hugging Face`.

Мы используем готовые веса `E5` для получения векторов. Усредняем представления токенов с учётом маски и нормализуем результат. На GPU модель работает в `float16`, а `DataParallel` распределяет кодирование между доступными видеокартами, максимум двумя.


In [2]:
MODEL = snapshot_download(repo_id=MODEL, local_dir=MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)
base_model = AutoModel.from_pretrained(MODEL, local_files_only=True)

if DEVICE.type == "cuda":
    base_model = base_model.half()

base_model = base_model.to(DEVICE).eval()


class E5Pooler(torch.nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, **batch):
        hidden = self.encoder(**batch).last_hidden_state.float()
        mask = batch['attention_mask'].unsqueeze(-1)
        return torch.nn.functional.normalize((hidden * mask).sum(1) / mask.sum(1), dim=1)


model = torch.nn.DataParallel(E5Pooler(base_model), device_ids=GPU_IDS)


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: /kaggle/working/models/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 3. Подготовка текстов и вспомогательные функции

Ниже собраны функции, которые понадобятся для поиска и обучения:

- `read` читает `Parquet`, заполняет пропуски в текстах и приводит числовые поля к нужному типу;
- `normalize` и `terms` очищают текст, убирают стоп-слова и приводят русские слова к основам;
- `embedding` строит векторы `E5` и сохраняет их в кэш `.npy`; если файл уже существует, загружает готовые векторы;
- `BM25` и `index` готовят индексы для поиска по словам и символьным фрагментам;
- `parameter` извлекает вид и тип услуги из текстовых параметров.

Отдельно задаём `ServiceNet` - небольшую нейросеть, которая по вектору запроса предсказывает вероятности микрокатегорий услуг. Она обучается на истории взаимодействий, а лучшие веса по валидационной ошибке сохраняются в `service_model.pt`.


In [3]:
def read(path):
    frame = pd.read_parquet(path)
    frame.columns = frame.columns.str.replace('_infn_', '_infm_')

    for name in frame:
        if name.endswith(('_raw', '_text')) or name == 'search_query':
            frame[name] = frame[name].fillna('').astype(str)

        elif name in ['item_price', 'item_latitude', 'item_longitude']:
            frame[name] = pd.to_numeric(frame[name], errors='coerce').astype(np.float32)

    return frame


def normalize(text):
    if text is None or pd.isna(text):
        return ''

    return ' '.join(
        re.findall(
            r'[a-zа-я0-9]+',
            unicodedata.normalize('NFKC', str(text)).lower().replace('ё', 'е'),
        )
    )


def embedding(texts, name, length=64):
    path = OUT / f"{name}.npy"

    if path.exists():
        return np.load(path)

    result = np.empty((len(texts), 768), dtype=np.float16)
    order = np.argsort([len(text) for text in texts], kind="stable")

    for start in tqdm(range(0, len(order), EMB_BATCH), desc=f"Embedding {name}"):
        ids = order[start:start + EMB_BATCH]

        batch = tokenizer(
            [texts[i] for i in ids],
            padding=True,
            truncation=True,
            max_length=length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.inference_mode():
            vector = model(**batch)

        result[ids] = vector.cpu().numpy()

    np.save(path, result)

    return result


def index(text, analyzer, grams):
    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=grams,
        min_df=2,
        max_features=180000,
        sublinear_tf=True,
        dtype=np.float32
    )

    return vectorizer, vectorizer.fit_transform(text).T


STOP = set('и в во на для по с со из к ко от до у о об а но или за под при это как что'.split())
STEMMER = RussianStemmer()

PARAM_KEYS = [
    'Вид услуги', 'Тип услуги автосервиса', 'Тип услуги', 'Место оказания услуг', 'Тип стоимости',
    'Работаете с', 'Опыт работы', 'Как вы работаете', 'Кто оказывает услуги',
    'Где вы оказываете услуги', 'Ваши клиенты', 'Специальность или сфера', 'Начальная цена',
    'Предоплата', 'Гарантия', 'Рабочие дни', 'Выезд к клиенту', 'Онлайн-запись',
    'Рейтинг пользователя', 'Берёте ли', 'Услуга', 'Куда выезжаете', 'Участие в пилоте',
    'Грузоподъёмность', 'Груз', 'Чем вы занимаетесь',
]

BOUNDARY = '|'.join(re.escape(k) for k in sorted(PARAM_KEYS, key=len, reverse=True))


@lru_cache(maxsize=300000)
def stem(token):
    return STEMMER.stem(token) if re.search('[а-я]', token) else token


def terms(text):
    return ' '.join(stem(t) for t in normalize(text).split() if t not in STOP)


def parameter(text, key):
    qualifier = r'(?!автосервиса\b)' if key == 'Тип услуги' else ''
    match = re.search(
        re.escape(key) + r'\s+' + qualifier + r'(.*?)(?=\s+(?:' + BOUNDARY + r')\b|$)',
        text,
    )
    value = match.group(1) if match else ''
    return '' if any(value.startswith(k) for k in PARAM_KEYS) else normalize(value)


class BM25:
    def __init__(self, docs, b=.65, max_features=250000):
        self.vectorizer = CountVectorizer(
            token_pattern=r'(?u)\b\w+\b', lowercase=False,
            max_features=max_features, dtype=np.float32,
        )
        counts = self.vectorizer.fit_transform(docs).tocsr()
        lengths = counts.sum(1).A.ravel()

        df = np.bincount(counts.indices, minlength=counts.shape[1])
        self.idf = np.log1p((len(docs) - df + .5) / (df + .5)).astype(np.float32)
        denominator = 1.2 * (1 - b + b * lengths / max(lengths.mean(), 1))
        counts.data = counts.data * 2.2 / (counts.data + np.repeat(denominator, np.diff(counts.indptr)))
        counts.data *= self.idf[counts.indices]
        self.index = counts.T.tocsr()

    def score(self, texts):
        q = self.vectorizer.transform(texts)
        q.data[:] = 1
        return (q @ self.index).tocsr()


class ServiceNet(torch.nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(768, 512), torch.nn.GELU(),
            torch.nn.Dropout(.15), torch.nn.Linear(512, classes),
        )

    def forward(self, x):
        return self.net(x)


def train_service(history, vectors, all_texts):
    torch.manual_seed(20260928)
    history = history.assign(norm=history.search_query.map(normalize))
    texts = sorted(history.norm.unique())
    categories = sorted(int(c) for c in history.item_microcat_id.unique())

    qi = {q: i for i, q in enumerate(texts)}
    ci = {c: i for i, c in enumerate(categories)}
    lookup = {q: i for i, q in enumerate(all_texts)}

    target = np.zeros((len(texts), len(categories)), np.float32)

    for (q, c), n in history.groupby(['norm', 'item_microcat_id']).size().items():
        target[qi[q], ci[c]] = np.log1p(n)
    target /= target.sum(1, keepdims=True)

    x = torch.tensor(
        np.array(vectors[[lookup[q] for q in texts]], dtype=np.float32), device='cuda:0'
    )
    y = torch.tensor(target, device='cuda:0')

    order = np.random.default_rng(20260928).permutation(len(texts))
    split = int(.9 * len(order))
    fit = torch.tensor(order[:split], device='cuda:0')
    validation = torch.tensor(order[split:], device='cuda:0')

    net = ServiceNet(len(categories)).cuda().train()
    optimizer = torch.optim.AdamW(net.parameters(), lr=.002, weight_decay=.01)
    best_loss = float('inf')

    for epoch in tqdm(range(30), desc='Service classifier'):
        net.train()
        shuffled = fit[torch.randperm(len(fit), device='cuda:0')]

        for b in shuffled.split(1024):
            loss = -(y[b] * torch.log_softmax(net(x[b] * 10), dim=1)).sum(1).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.inference_mode():
            loss = torch.cat([
                -(y[b] * torch.log_softmax(net(x[b] * 10), dim=1)).sum(1)
                for b in validation.split(2048)
            ]).mean().item()

        if loss < best_loss:
            best_loss = loss
            best = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}

    result = dict(state=best, categories=categories, validation_loss=best_loss)
    torch.save(result, OUT / 'service_model.pt')
    print('Service validation loss:', best_loss)

    return result


### 4. Поиск кандидатов и признаки

Для каждого запроса собираем объявления из нескольких поисковых выдач и удаляем повторы. `E5` сравнивает смысл запроса с заголовком и описанием, `BM25` ищет совпадения слов, а символьный `TF-IDF` - похожие фрагменты заголовков.

Дополнительно используем историю 40 ближайших по смыслу запросов, географию и прогноз классификатора услуг. Для каждой пары `запрос` - `объявление` получаем **54 признака**: текстовую близость, совпадение локации и категории, расстояние, рейтинг, цену, популярность и соответствие фильтрам.

Для `E5` к запросам добавляем префикс `query:`, а к объявлениям - `passage:`. Заголовки и запросы ограничиваем 64 токенами, заголовок с описанием - 256.

В конце ячейки определяем `top`, `select` и `recall`: они выбирают лучшие объявления и считают среднюю долю найденных положительных ответов. При выборе 50 объявлений это наша метрика `Recall@50`.


In [4]:
def candidates(queries, items, history, limit):
    OUT.mkdir(exist_ok=True)
    titles = items.item_title_raw.tolist()

    body = ['passage: ' + t + '. ' + d for t, d in zip(titles, items.item_description_raw)]
    split = (len(items) + 1) // 2
    dense_np = embedding(body, 'items_e5', 256)
    title_np = embedding(['passage: ' + t for t in titles], 'titles_e5')

    dense = [
        torch.from_numpy(dense_np[:split]).to("cuda:0").T.contiguous(),
        torch.from_numpy(dense_np[split:]).to(SECOND_DEVICE).T.contiguous(),
    ]

    title = [
        torch.from_numpy(title_np[:split]).to("cuda:0").T.contiguous(),
        torch.from_numpy(title_np[split:]).to(SECOND_DEVICE).T.contiguous(),
    ]

    texts = sorted(
        set(read(DATA / 'train.parquet').search_query.map(normalize))
        | set(read(DATA / 'benchmark_queries.parquet').search_query.map(normalize))
        | set(queries.search_query.map(normalize))
    )

    vectors = embedding(['query: ' + text for text in texts], 'queries_e5')
    lookup = {text: i for i, text in enumerate(texts)}

    n = len(items)
    ids = items.item_id.to_numpy()
    idmap = {v: i for i, v in enumerate(ids)}

    locations, loc_idx = np.unique(items.item_location_id, return_inverse=True)
    loc_map = {int(v): i for i, v in enumerate(locations)}

    microcats, micro_idx = np.unique(items.item_microcat_id, return_inverse=True)
    micro_map = {int(v): i for i, v in enumerate(microcats)}

    lat, lon = items.item_latitude.to_numpy(), items.item_longitude.to_numpy()
    title_norm = items.item_title_raw.map(normalize).to_numpy()
    title_terms = items.item_title_raw.map(terms)
    title_terms = [set(t.split()) for t in title_terms]
    title_length = items.item_title_raw.str.len().to_numpy(np.float32)
    desc_length = items.item_description_raw.str.len().to_numpy(np.float32)

    print('Building lexical index', flush=True)
    title_bm25 = BM25(items.item_title_raw.map(terms), b=.4)
    body = items.item_title_raw + ' ' + items.item_description_raw
    body_norm = body.map(normalize).to_numpy()
    body_bm25 = BM25(body.map(terms))
    filter_bm25 = BM25(items.item_infm_params_text.map(terms), b=.3, max_features=120000)
    char, char_index = index(items.item_title_raw.map(normalize), 'char_wb', (3, 5))
    char_index = char_index.tocsr()
    body_binary = body_bm25.index.copy()
    body_binary.data[:] = 1

    service_type = items.item_infm_params_text.map(lambda s: parameter(s, 'Вид услуги')).to_numpy()
    service_subtype = items.item_infm_params_text.map(lambda s: parameter(s, 'Тип услуги')).to_numpy()

    names = [
        'item_rating', 'item_rating_reviews_count', 'item_price',
        'item_is_phone_hidden', 'item_is_message_forbidden', 'item_category_id',
    ]
    meta = {k: items[k].to_numpy(np.float32) for k in names}

    saved = (
        torch.load(OUT / 'service_model.pt', map_location='cpu', weights_only=True)
        if 'query_id' in queries else train_service(history, vectors, texts)
    )
    service_net = ServiceNet(len(saved['categories'])).to(DEVICE).eval().requires_grad_(False)
    service_net.load_state_dict(saved['state'])
    cat_map = {v: i for i, v in enumerate(saved['categories'])}
    service_cols = np.array([cat_map.get(int(c), -1) for c in microcats])

    norms = history.search_query.map(normalize)
    texts = sorted(norms.unique())
    qmap = {q: i for i, q in enumerate(texts)}
    query_vocab = set(qmap)
    qr = norms.map(qmap).to_numpy()
    hist_dense = torch.as_tensor(
        vectors[[lookup[q] for q in texts]], device=DEVICE
    ).T.contiguous()

    cat = history.item_microcat_id.map(micro_map)
    valid = cat.notna().to_numpy()
    qcats = sp.csr_matrix(
        (np.ones(valid.sum(), np.float32), (qr[valid], cat[valid].astype(int))),
        shape=(len(texts), len(microcats)),
    )
    qcats = sp.diags(1 / np.maximum(qcats.sum(1).A.ravel(), 1)) @ qcats

    ix = history.item_id.map(idmap)
    valid = ix.notna().to_numpy()
    qitems = sp.csr_matrix(
        (np.ones(valid.sum(), np.float32), (qr[valid], ix[valid].astype(int))),
        shape=(len(texts), n),
    )
    qitems.data = np.log1p(qitems.data)

    popularity = np.log1p(
        history.item_id.value_counts().reindex(ids, fill_value=0).to_numpy()
    ).astype(np.float32)
    count = np.bincount(qr, minlength=len(texts)).astype(np.float32)
    local = np.bincount(
        qr, weights=(history.search_location_id == history.item_location_id),
        minlength=len(texts),
    )
    query_local = (local + 4) / (count + 5)

    counts = history.groupby(['search_location_id', 'item_location_id']).size()
    loc_counts = {}
    for location, group in counts.groupby(level=0):
        arr = np.zeros(len(locations), np.float32)
        for (_, item_location), count in group.items():
            if item_location in loc_map:
                arr[loc_map[item_location]] = count
        loc_counts[location] = arr

    global_loc = (
        history.item_location_id.value_counts().reindex(locations, fill_value=0).to_numpy(np.float32) + 1
    )
    global_loc /= global_loc.sum()

    cols = ['item_latitude', 'item_longitude']
    city = history.groupby('item_location_id')[cols].median().combine_first(
        items.groupby('item_location_id')[cols].median()
    )
    centers = {int(k): tuple(v) for k, v in city.iterrows()}
    regions = {
        int(k): tuple(v)
        for k, v in history.groupby('search_location_id')[cols].median().iterrows()
    }

    @lru_cache(maxsize=24)
    def geo_features(location):
        count = loc_counts.get(location, np.zeros(len(locations), np.float32))
        prob = (count + 10 * global_loc) / (count.sum() + 10)
        lift = np.log(np.maximum(prob / global_loc, 1e-5))[loc_idx]
        exact = (items.item_location_id.to_numpy() == location).astype(np.float32)
        center = centers.get(location, regions.get(location))
        known = int(center is not None and np.isfinite(center).all())

        if known:
            lat_rad, lon_rad, lat0, lon0 = map(np.radians, [lat, lon, *center])
            a = (
                np.sin((lat_rad - lat0) / 2) ** 2
                + np.cos(lat_rad) * np.cos(lat0) * np.sin((lon_rad - lon0) / 2) ** 2
            )
            distance = np.nan_to_num(
                6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1))), nan=1000
            ).astype(np.float32)
        else:
            distance = np.full(n, 1000, np.float32)

        bias = (.018 * np.clip(lift, -6, 6) + .055 * exact).astype(np.float32)

        return exact, np.log1p(distance), lift, prob[loc_idx], bias, known

    choose = lambda score, k: top(score[:size], k)
    norm = lambda score: score / max(float(score[:size].max()), 1e-6)

    for start in tqdm(range(0, len(queries), QUERY_BATCH), desc='Ranking'):
        batch = queries.iloc[start:start + QUERY_BATCH]
        qnorm = batch.search_query.map(normalize).tolist()
        qt = batch.search_query.map(terms).tolist()

        scores = dict(
            bm25_title=title_bm25.score(qt),
            bm25_body=body_bm25.score(qt),
            char_title=(char.transform(qnorm) @ char_index).tocsr(),
            bm25_filter=filter_bm25.score(batch.search_infm_params_text.map(terms))
        )

        counts = body_bm25.vectorizer.transform(qt)
        counts.data[:] = 1

        coverage = (counts @ body_binary).toarray()
        weighted = counts.multiply(body_bm25.idf)
        idf_coverage = (weighted @ body_binary).toarray() / np.maximum(weighted.sum(1).A, 1e-6)

        qarray = vectors[[lookup[q] for q in qnorm]]
        qvec = torch.as_tensor(qarray, device=DEVICE)
        q1 = torch.as_tensor(qarray, device=SECOND_DEVICE)

        semantic0, heading0 = qvec @ dense[0], qvec @ title[0]
        semantic1, heading1 = q1 @ dense[1], q1 @ title[1]

        values, nn = torch.topk(qvec @ hist_dense, min(40, hist_dense.shape[1]), dim=1)
        services = service_net(
            torch.as_tensor(qarray, device=DEVICE, dtype=torch.float32) * 10
        ).softmax(1).cpu().numpy()
        body_scores = np.concatenate(
            [semantic0.float().cpu().numpy(), semantic1.float().cpu().numpy()], axis=1
        )
        headings = np.concatenate(
            [heading0.float().cpu().numpy(), heading1.float().cpu().numpy()], axis=1
        )
        values, nn = values.float().cpu().numpy(), nn.cpu().numpy()

        for j, row in enumerate(batch.itertuples()):
            size = limit if 'query_id' in queries or getattr(row, 'split', '') == 'dev_catalog' else n
            s = {k: v.getrow(j).toarray().ravel() for k, v in scores.items()}
            exact, distance, lift, locprob, geo, known = geo_features(row.search_location_id)

            weights = np.exp(35 * (values[j] - values[j, 0])).astype(np.float32)
            weights /= weights.sum()

            catprob = np.asarray(weights @ qcats[nn[j]]).ravel().astype(np.float32)
            hist = np.asarray(
                (weights * np.exp(10 * (values[j] - .8))) @ qitems[nn[j]]
            ).ravel().astype(np.float32)

            cat_clf = np.zeros(len(microcats), np.float32)
            valid = service_cols >= 0
            cat_clf[valid] = services[j, service_cols[valid]]
            item_clf = cat_clf[micro_idx]

            sem, ts = body_scores[j], headings[j]
            lexical = (
                .46 * norm(s['bm25_title']) + .32 * norm(s['bm25_body']) + .22 * norm(s['char_title'])
            )
            hybrid = .58 * sem + .42 * ts + .10 * lexical + geo

            pool = np.unique(np.concatenate([
                choose(score, k)
                for score, k in [
                    (hybrid, 400),
                    (sem + geo, 250),
                    (ts + geo, 170),
                    (hybrid + .06 * norm(s['bm25_filter']), 150),
                    (lexical + 2 * geo, 230),
                    (norm(s['bm25_body']) + 2 * geo, 140),
                    (sem, 100),
                    (ts, 60),
                    (norm(hist) + 2 * geo, 100),
                    (hybrid + .035 * np.log(np.maximum(item_clf, 1e-4)), 160),
                    (.6 * sem + .4 * ts + geo + .10 * idf_coverage[j], 130),
                ]
            ]))

            p = pool
            tokens = set(qt[j].split())
            denom = max(len(tokens), 1)
            service = parameter(row.search_infm_params_text, 'Вид услуги')
            subtype = parameter(row.search_infm_params_text, 'Тип услуги')
            has_rating = 'Рейтинг пользователя' in row.search_infm_params_text
            nums = set(re.findall(r'\b\d+\b', qnorm[j]))

            data = {k: v[p] for k, v in s.items()}

            data.update(
                e5_body=sem[p],
                e5_title=ts[p],
                lexical=lexical[p],
                hybrid=hybrid[p],
                history=hist[p],
                history_normalized=norm(hist)[p],
                semantic_category=catprob[micro_idx[p]],
                exact_location=exact[p],
                log_distance_km=distance[p],
                location_lift=lift[p],
                location_probability=locprob[p],
                geo_bias=geo[p],
                popularity=popularity[p],
                rating=meta['item_rating'][p],
                log_reviews=np.log1p(meta['item_rating_reviews_count'][p]),
                log_price=np.log1p(np.maximum(meta['item_price'][p], 0)),
                phone_hidden=meta['item_is_phone_hidden'][p],
                message_forbidden=meta['item_is_message_forbidden'][p],
                category_matches=(
                    (meta['item_category_id'][p] == row.search_category) | (row.search_category == 0)
                ).astype(np.float32),
                microcategory=micro_idx[p].astype(np.float32),
                title_length=title_length[p],
                description_length=desc_length[p],
                title_coverage=np.array([len(tokens & title_terms[i]) / denom for i in p]),
                title_precision=np.array([
                    len(tokens & title_terms[i]) / max(len(title_terms[i]), 1) for i in p
                ]),
                query_in_title=np.array([qnorm[j] in title_norm[i] for i in p], np.float32)
            )

            for name in ['bm25_title', 'bm25_body', 'bm25_filter', 'char_title']:
                data[name + '_normalized'] = norm(s[name])[p]

            scalars = dict(
                nearest_query_similarity=values[j, 0],
                query_locality=float(weights @ query_local[nn[j]]),
                geo_known=known,
                query_tokens=len(tokens),
                query_length=len(qnorm[j]),
                has_filter=bool(row.search_infm_params_text),
                query_seen=qnorm[j] in query_vocab,
                query_category=row.search_category,
            )

            data.update({k: np.full(len(p), v, np.float32) for k, v in scalars.items()})

            data.update(
                body_term_coverage=coverage[j, p] / denom,
                body_idf_coverage=idf_coverage[j, p],
                body_all_terms=(coverage[j, p] >= len(tokens)).astype(np.float32),
                query_in_body=np.array([qnorm[j] in body_norm[i] for i in p], np.float32),
                service_probability=item_clf[p],
                service_probability_relative=item_clf[p] / max(float(services[j].max()), 1e-6),
                service_type_match=np.array([
                    int(service_type[i] == service) if service and service_type[i] else -1
                    for i in p
                ]),
                service_subtype_match=np.array([
                    int(service_subtype[i] == subtype) if subtype and service_subtype[i] else -1
                    for i in p
                ]),
                rating_filter_pass=np.where(
                    has_rating, meta['item_rating'][p] >= 4, -1
                ).astype(np.float32),
                numbers_in_title=np.array([
                    sum(n in title_norm[i].split() for n in nums) / len(nums) if nums else -1
                    for i in p
                ]),
                semantic_gap_body=sem[p] - sem[:size].max(),
                semantic_gap_title=ts[p] - ts[:size].max(),
                service_entropy=np.full(
                    len(p), -np.sum(services[j] * np.log(np.maximum(services[j], 1e-9)))
                ),
            )

            yield np.column_stack([data[k] for k in FEATURES]).astype(np.float32), pool


def top(scores, k=50):
    k = min(k, len(scores))

    if not k:
        return np.empty(0, np.int32)

    threshold = np.partition(scores, len(scores) - k)[len(scores) - k]
    above = np.flatnonzero(scores > threshold)
    ids = np.concatenate([above, np.flatnonzero(scores == threshold)[:k - len(above)]])

    return ids[np.lexsort((ids, -scores[ids]))].astype(np.int32)


def select(model, blocks):
    scores = model.predict(np.concatenate([x for x, ids in blocks]), num_threads=6)

    offset = 0

    for x, ids in blocks:
        yield ids[top(scores[offset:offset + len(ids)])]
        offset += len(ids)


def recall(predictions, truth):
    predictions = list(predictions)

    return float(np.mean([
        len(set(p) & set(t)) / len(set(t)) for p, t in zip(predictions, truth)
    ]))


### 5. Разбиение данных и ранжировщик

Разделяем данные по нормализованному тексту запроса: 12 000 - в `train`, 1 500 - в `dev`, 1 000 - в `dev_catalog` и 1 500 - в `test`. Эти тексты исключаем из истории, по которой строятся признаки и обучается классификатор услуг.

Для локального обучения добавляем в каталог положительные объявления из истории. На `dev_catalog` поиск ограничиваем исходным каталогом. У части обучающих запросов убираем категорию или фильтры, чтобы модель видела разные варианты запроса.

Ранжировщик - `LightGBM` с функцией потерь `LambdaRank`. Каждый запрос образует группу кандидатов: известным положительным объявлениям ставим метку 1, остальным - 0. Веса групп учитывают долю запросов с фильтрами в бенчмарке.


In [5]:
def prepare(train, original):
    train = train.assign(norm=train.search_query.map(normalize))

    eligible = train[train.item_id.isin(original.item_id)]
    texts = sorted(eligible.norm.unique())

    np.random.default_rng(SPLIT_SEED).shuffle(texts)

    base = eligible[eligible.norm.isin(texts[:5000])].drop_duplicates(SEARCH).sample(
        frac=1, random_state=SPLIT_SEED
    )
    base = (
        base.drop_duplicates('norm').set_index('norm')
        .loc[texts[:5000]].reset_index()[['norm'] + SEARCH]
    )
    base['split'] = ['train'] * 3000 + ['dev_catalog'] * 1000 + ['train'] * 1000

    rng = np.random.default_rng(SEED)

    rest = sorted(set(train.norm) - set(base.norm))
    rng.shuffle(rest)

    extra = train[train.norm.isin(rest[:11000])].drop_duplicates(SEARCH).sample(
        frac=1, random_state=SEED
    )
    extra = (
        extra.drop_duplicates('norm').set_index('norm')
        .loc[rest[:11000]].reset_index()[['norm'] + SEARCH]
    )
    extra['split'] = ['train'] * 8000 + ['dev'] * 1500 + ['test'] * 1500

    queries = pd.concat([base, extra], ignore_index=True).reset_index(names='row')

    labels = queries.merge(train[SEARCH + ['item_id']], on=SEARCH).groupby('row').item_id.agg(
        lambda s: sorted(set(s))
    )

    known = set(original.item_id)

    labels = [
        [i for i in ids if i in known] if part == 'dev_catalog' else ids
        for ids, part in zip(labels, queries.split)
    ]

    extra_ids = {i for ids in labels for i in ids} - known

    items = pd.concat(
        [
            original,
            train[train.item_id.isin(extra_ids)].drop_duplicates('item_id')[original.columns],
        ],
        ignore_index=True
    )

    index = {item: i for i, item in enumerate(items.item_id)}

    queries['positives'] = [[index[i] for i in ids] for ids in labels]

    augmentation = rng.random(len(queries))

    queries.loc[queries.split.eq('train') & (augmentation < .10), 'search_category'] = 0

    queries.loc[
        queries.split.eq('train') & (augmentation >= .10) & (augmentation < .22),
        'search_infm_params_text'
    ] = ''

    history = train[~train.norm.isin(queries.norm)]

    return queries, items, history


def fit(blocks, labels, trees):
    x = np.concatenate([x for x, ids in blocks])
    y = np.concatenate([
        np.isin(ids, truth).astype(np.int8) for (_, ids), truth in zip(blocks, labels)
    ])
    groups = np.array([len(ids) for x, ids in blocks], np.int32)
    offsets = np.r_[0, np.cumsum(groups)[:-1]]

    filtered = x[offsets, FEATURES.index('has_filter')] > 0
    share = float(filtered.mean())
    target = float(read(DATA / 'benchmark_queries.parquet').search_infm_params_text.ne('').mean())
    weights = np.where(filtered, target / max(share, 1e-6), (1 - target) / max(1 - share, 1e-6))

    model = lgb.LGBMRanker(
        objective='lambdarank', n_estimators=trees, learning_rate=.045,
        num_leaves=31, min_child_samples=120, reg_lambda=8, reg_alpha=.15,
        colsample_bytree=.85, subsample=.85, subsample_freq=1, force_col_wise=True,
        lambdarank_truncation_level=60, n_jobs=6, random_state=SEED, verbosity=-1,
    )

    def progress(env):
        if (env.iteration + 1) % 100 == 0:
            print('Trees:', env.iteration + 1, flush=True)

    model.fit(
        x, y, group=groups, feature_name=FEATURES,
        categorical_feature=['microcategory', 'query_category'],
        sample_weight=np.repeat(weights, groups).astype(np.float32), callbacks=[progress],
    )

    return model.booster_


def subset(blocks, queries, indices):
    return [blocks[i] for i in indices], queries.positives.iloc[indices].tolist()


def development_recall(predictions, truth, contexts):
    values = np.array([len(set(p) & set(t)) / len(set(t)) for p, t in zip(predictions, truth)])
    filtered = contexts.search_infm_params_text.ne('').to_numpy()
    target = float(read(DATA / 'benchmark_queries.parquet').search_infm_params_text.ne('').mean())

    return float(target * values[filtered].mean() + (1 - target) * values[~filtered].mean())


### 6. Обучение и подготовка ответа

Функция `train` готовит данные и кандидатов, затем обучает ранжировщик на 650 деревьях. Проверяем варианты из 100, 200, 350, 500 и 650 деревьев и выбираем лучший по взвешенному `Recall@50`: 70% оценки даёт `dev`, 30% - `dev_catalog`.

После выбора считаем качество на `test`, сохраняем его в `metrics.json` и переобучаем ранжировщик на всех подготовленных группах. Отдельная метрика `candidate_recall` показывает, сколько положительных объявлений удалось найти до отбора лучших 50.

Функция `predict` использует всю историю из `train.parquet`, загружает обученные модели и подбирает 50 объявлений для каждого запроса бенчмарка. Перед сохранением проверяем уникальность идентификаторов и принадлежность объявлений исходному каталогу.


In [6]:
def train():
    queries, items, history = prepare(
        read(DATA / 'train.parquet'),
        read(DATA / 'benchmark_items.parquet')
    )

    items.to_parquet(OUT / 'corpus.parquet', index=False)

    blocks = list(candidates(
        queries, items, history, len(read(DATA / 'benchmark_items.parquet'))
    ))

    parts = {
        name: np.flatnonzero(queries.split.isin(names))
        for name, names in [
            ('train', ['train']),
            ('dev', ['dev']),
            ('dev_catalog', ['dev_catalog']),
            ('test', ['test'])
        ]
    }

    model = fit(*subset(blocks, queries, parts['train']), 650)
    trials = {}

    for trees in [100, 200, 350, 500, 650]:
        limited = lgb.Booster(model_str=model.model_to_string(num_iteration=trees))
        values = {}

        for name in ['dev', 'dev_catalog']:
            development, truth = subset(blocks, queries, parts[name])
            values[name] = development_recall(
                select(limited, development), truth, queries.iloc[parts[name]]
            )

        trials[trees] = .7 * values['dev'] + .3 * values['dev_catalog']

    trees = max(trials, key=lambda t: (trials[t], -t))
    model = lgb.Booster(model_str=model.model_to_string(num_iteration=trees))

    test, truth = subset(blocks, queries, parts['test'])
    result = {
        'selected_trees': trees,
        'dev_recall': trials,
        'test_queries': len(test),
        'recall_at_50': recall(select(model, test), truth),
        'candidate_recall': recall([ids for x, ids in test], truth),
        'external_recall_at_50': None
    }

    print(result, flush=True)

    (OUT / 'metrics.json').write_text(json.dumps(result, indent=2))

    order = np.concatenate([parts[name] for name in ['train', 'dev_catalog', 'dev', 'test']])
    fit(*subset(blocks, queries, order), trees).save_model(str(OUT / 'ranker.txt'))


def predict():
    items, queries = read(OUT / 'corpus.parquet'), read(DATA / 'benchmark_queries.parquet')
    original = read(DATA / 'benchmark_items.parquet')

    blocks = list(candidates(queries, items, read(DATA / 'train.parquet'), len(original)))
    model = lgb.Booster(model_file=str(OUT / 'ranker.txt'))

    predictions = [original.item_id.iloc[ids].tolist() for ids in select(model, blocks)]

    answer = pd.DataFrame({
        'query_id': queries.query_id,
        'answer': [' '.join(top50) for top50 in predictions],
    })
    answer.to_csv('/kaggle/working/answer.csv', index=False)


### 7. Запуск обучения и предсказания

Теперь запускаем `train()`, затем `predict()`. Ячейка выводит время каждого этапа и путь к готовому ответу: `/kaggle/working/answer.csv`.

**P.S.** Сохранённые ниже логи относятся к прежнему запуску с другим набором проверяемых деревьев. Качество текущей версии нужно смотреть в новом `metrics.json` после повторного выполнения.


In [7]:
start = time.time()

print("=== 1/2 TRAIN ===", flush=True)
train()
print(f"TRAIN готов за {(time.time() - start) / 60:.1f} мин", flush=True)

pred_start = time.time()

print("\n=== 2/2 PREDICT ===", flush=True)
predict()
print(f"PREDICT готов за {(time.time() - pred_start) / 60:.1f} мин", flush=True)

print(f"\nВСЁ ГОТОВО за {(time.time() - start) / 60:.1f} мин")
print("/kaggle/working/answer.csv")


=== 1/2 TRAIN ===


Embedding items_e5:   0%|          | 0/786 [00:00<?, ?it/s]

Embedding titles_e5:   0%|          | 0/786 [00:00<?, ?it/s]

Embedding queries_e5:   0%|          | 0/294 [00:00<?, ?it/s]

Building lexical index


...

Service validation loss: 1.4230687618255615


Ranking:   0%|          | 0/63 [00:00<?, ?it/s]

Trees: 100
Trees: 200
Trees: 300
Trees: 400
Trees: 500
Trees: 600
{'selected_trees': 650, 'dev_recall': {100: 0.9382717311425635, 200: 0.9455914699753599, 350: 0.9445523869934738, 500: 0.9448013847771644, 650: 0.9466967947756275}, 'test_queries': 1500, 'recall_at_50': 0.9425555555555557, 'candidate_recall': 0.981, 'external_recall_at_50': None}
Trees: 100
Trees: 200
Trees: 300
Trees: 400
Trees: 500
Trees: 600
TRAIN готов за 82.1 мин

=== 2/2 PREDICT ===
Building lexical index


Ranking:   0%|          | 0/10 [00:00<?, ?it/s]

PREDICT готов за 6.5 мин

ВСЁ ГОТОВО за 88.7 мин
/kaggle/working/answer.csv


### 8. Проверка answer.csv

Открываем готовый файл и смотрим на результат. В нём две колонки:

- `query_id` - идентификатор поискового запроса;
- `answer` - 50 уникальных `item_id`, разделённых пробелами.

Каждая строка соответствует одному запросу из `benchmark_queries.parquet`.


In [8]:
answer = pd.read_csv("/kaggle/working/answer.csv")
answer


,query_id,answer
0,70DfDUpwjxB4lzFd,14d535285758af89 d01e14d2ffd4ddba deb58d7ad8c2...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 cbeccbecb1fb8d86 1344796085e9...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 dab52187b4500d9b 3f89b8062dc8...
3,660ac9QVtXkRxZC3,67b2cd847669eb36 4c15e7abba063ac5 af91ec4a29b6...
4,YgHcM9MVbxKnxD1e,9c41fe789f98d702 ba78ad593ec3412d 1a6263439454...
...,...,...
2447,JYUbsR4EU5RtfGUA,854821d3c255d674 bd1e805a69773628 187f534b3c39...
2448,SuCUWvKrKFRKTKuT,477aea95b877581b 8b11d923da92aa6e 354957b0b9e6...
2449,fWIRLM5wCElVVjRV,98c8c98bbb109f15 620de64d18a1c0e9 18c330f57fbd...
2450,VjX4ayHpKHlftRgu,5d879e98f19d057b d9984c0fb49f6e79 6aef0c1ede43...


### 9. Результат

- Используем готовую `E5` для поиска по смыслу и `BM25` с символьным `TF-IDF` для поиска по текстовым совпадениям.
- Обучаем классификатор услуг и ранжировщик `LightGBM` на 54 признаках.
- Проверяем локальное качество через `Recall@50` и сохраняем метрики в `/kaggle/working/artifacts/metrics.json`.
- Сохраняем модели в `artifacts/service_model.pt` и `artifacts/ranker.txt`, а итоговый ответ - в `/kaggle/working/answer.csv`.

Локальный `Recall@50` считается на нашем разбиении данных. Внешняя оценка на скрытом бенчмарке в `metrics.json` не задана.
